# 📊 SQL Dashboard Queries

## Purpose

This notebook contains **ready-to-use SQL queries** for creating dashboards and visualizations.

All queries read from the **Gold layer** analytics tables, which are:
* Pre-aggregated for fast performance
* Optimized with Delta Lake
* Business-ready with clean metrics

## Dashboard Use Cases:
1. Executive summary - Overall Netflix content metrics
2. Content strategy - Genre and type trends
3. Geographic insights - Country distribution
4. Content planning - Duration and rating analysis

## How to Use:
1. Run any query cell below
2. Click **+ Visualization** on the results
3. Choose chart type (bar, line, pie, etc.)
4. Configure axes and labels
5. Add to a dashboard

---

**Note:** These queries are designed to be beginner-friendly with clear column names and comprehensive comments.

In [0]:
%sql
-- ===================================================================
-- DASHBOARD QUERY 1: Top 20 Genres
-- ===================================================================
-- Business Question: What are Netflix's most popular content genres?
--
-- Visualization Suggestion: 
--   - Horizontal Bar Chart
--   - X-axis: content_count
--   - Y-axis: genre
--   - Color by: percentage
-- ===================================================================

SELECT 
  genre,
  content_count,
  movie_count,
  tv_show_count,
  percentage,
  avg_content_age,
  avg_release_year
FROM workspace.netflix_gold.top_genres
ORDER BY content_count DESC
LIMIT 20;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- ===================================================================
-- DASHBOARD QUERY 2: Content Growth Trend
-- ===================================================================
-- Business Question: How has Netflix content library grown year over year?
--
-- Visualization Suggestion:
--   - Line Chart (dual axis)
--   - X-axis: year_added
--   - Y-axis 1: total_content (line)
--   - Y-axis 2: cumulative_content (area)
-- ===================================================================

SELECT 
  year_added,
  total_content,
  movies_added,
  tv_shows_added,
  cumulative_content,
  cumulative_movies,
  cumulative_tv_shows
FROM workspace.netflix_gold.content_trend_by_year
WHERE year_added >= 2010  -- Focus on recent years
ORDER BY year_added;

In [0]:
%sql
-- ===================================================================
-- DASHBOARD QUERY 3: Movies vs TV Shows Over Time
-- ===================================================================
-- Business Question: Is Netflix shifting toward TV shows or movies?
--
-- Visualization Suggestion:
--   - Stacked Area Chart or Line Chart
--   - X-axis: year_added
--   - Y-axis: count
--   - Series: movies_added, tv_shows_added
-- ===================================================================

SELECT 
  year_added,
  movies_added,
  tv_shows_added,
  ROUND(movies_added * 100.0 / total_content, 1) as movie_percentage,
  ROUND(tv_shows_added * 100.0 / total_content, 1) as tv_show_percentage
FROM workspace.netflix_gold.content_trend_by_year
WHERE year_added >= 2015
ORDER BY year_added;

In [0]:
%sql
-- ===================================================================
-- DASHBOARD QUERY 4: Top Content Producing Countries
-- ===================================================================
-- Business Question: Which countries produce the most Netflix content?
--
-- Visualization Suggestion:
--   - Horizontal Bar Chart or World Map
--   - X-axis: content_count
--   - Y-axis: country_individual
--   - Color: percentage
-- ===================================================================

SELECT 
  country_individual as country,
  content_count,
  movie_count,
  tv_show_count,
  percentage,
  avg_content_age
FROM workspace.netflix_gold.content_by_country
ORDER BY content_count DESC
LIMIT 15;

In [0]:
%sql
-- ===================================================================
-- DASHBOARD QUERY 5: Content Ratings Distribution
-- ===================================================================
-- Business Question: What's the rating distribution? Family-friendly vs mature?
--
-- Visualization Suggestion:
--   - Pie Chart or Donut Chart
--   - Values: content_count
--   - Labels: rating
--   - Show percentage labels
-- ===================================================================

SELECT 
  rating,
  content_count,
  movie_count,
  tv_show_count,
  percentage,
  CASE 
    WHEN rating IN ('G', 'PG', 'TV-Y', 'TV-Y7', 'TV-G', 'TV-PG') THEN 'Family Friendly'
    WHEN rating IN ('PG-13', 'TV-14') THEN 'Teen'
    WHEN rating IN ('R', 'NC-17', 'TV-MA') THEN 'Mature'
    ELSE 'Other'
  END as rating_category
FROM workspace.netflix_gold.content_by_rating
ORDER BY content_count DESC;

In [0]:
%sql
-- ===================================================================
-- DASHBOARD QUERY 6: Movie Duration by Genre
-- ===================================================================
-- Business Question: How long are movies in different genres?
--
-- Visualization Suggestion:
--   - Bar Chart
--   - X-axis: genre
--   - Y-axis: avg_duration
--   - Filter: Only Movies
-- ===================================================================

SELECT 
  genre,
  content_count,
  avg_duration as avg_minutes,
  min_duration as min_minutes,
  max_duration as max_minutes
FROM workspace.netflix_gold.duration_by_genre
WHERE type = 'Movie'
  AND content_count >= 10  -- Only genres with sufficient data
ORDER BY content_count DESC
LIMIT 20;

In [0]:
%sql
-- ===================================================================
-- DASHBOARD QUERY 7: TV Show Length by Genre
-- ===================================================================
-- Business Question: How many seasons do TV shows typically have?
--
-- Visualization Suggestion:
--   - Bar Chart
--   - X-axis: genre
--   - Y-axis: avg_duration (seasons)
-- ===================================================================

SELECT 
  genre,
  content_count,
  avg_duration as avg_seasons,
  min_duration as min_seasons,
  max_duration as max_seasons
FROM workspace.netflix_gold.duration_by_genre
WHERE type = 'TV Show'
  AND content_count >= 5
ORDER BY avg_duration DESC
LIMIT 20;

In [0]:
%sql
-- ===================================================================
-- DASHBOARD QUERY 8: Content Distribution by Decade
-- ===================================================================
-- Business Question: What era of content is most represented?
--
-- Visualization Suggestion:
--   - Bar Chart or Area Chart
--   - X-axis: decade
--   - Y-axis: content_count
--   - Stacked by: movie_count and tv_show_count
-- ===================================================================

SELECT 
  decade,
  CONCAT(decade, 's') as decade_label,
  content_count,
  movie_count,
  tv_show_count,
  percentage,
  avg_content_age
FROM workspace.netflix_gold.content_by_decade
WHERE decade >= 1970  -- Focus on modern content
ORDER BY decade DESC;

In [0]:
%sql
-- ===================================================================
-- DASHBOARD QUERY 9: Genre Diversity - Movies vs TV Shows
-- ===================================================================
-- Business Question: Which content type has more genre diversity?
--
-- Visualization Suggestion:
--   - Scatter Plot
--   - X-axis: movie_count
--   - Y-axis: tv_show_count
--   - Size: content_count
--   - Labels: genre
-- ===================================================================

SELECT 
  genre,
  content_count,
  movie_count,
  tv_show_count,
  ROUND(movie_count * 100.0 / content_count, 1) as movie_percentage,
  ROUND(tv_show_count * 100.0 / content_count, 1) as tv_show_percentage,
  CASE 
    WHEN movie_count > tv_show_count * 2 THEN 'Movie Dominant'
    WHEN tv_show_count > movie_count * 2 THEN 'TV Show Dominant'
    ELSE 'Balanced'
  END as content_type_preference
FROM workspace.netflix_gold.top_genres
WHERE content_count >= 20  -- Only significant genres
ORDER BY content_count DESC
LIMIT 30;

Databricks data profile. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- ===================================================================
-- DASHBOARD QUERY 10: Executive Summary - Key Metrics
-- ===================================================================
-- Business Question: What are the top-level Netflix content metrics?
--
-- Visualization Suggestion:
--   - KPI Cards / Scorecards
--   - Display as individual metrics
-- ===================================================================

WITH 
  total_metrics AS (
    SELECT 
      COUNT(*) as total_titles,
      SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) as total_movies,
      SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) as total_tv_shows,
      COUNT(DISTINCT country) as unique_countries,
      ROUND(AVG(content_age), 1) as avg_content_age
    FROM workspace.netflix_silver.clean_titles
  ),
  genre_count AS (
    SELECT COUNT(*) as total_genres
    FROM workspace.netflix_gold.top_genres
  ),
  recent_additions AS (
    SELECT SUM(total_content) as added_last_year
    FROM workspace.netflix_gold.content_trend_by_year
    WHERE year_added = 2021  -- Change to latest available year
  )

SELECT 
  t.total_titles,
  t.total_movies,
  t.total_tv_shows,
  ROUND(t.total_movies * 100.0 / t.total_titles, 1) as movie_percentage,
  ROUND(t.total_tv_shows * 100.0 / t.total_titles, 1) as tv_show_percentage,
  t.unique_countries,
  g.total_genres,
  t.avg_content_age,
  r.added_last_year
FROM total_metrics t
CROSS JOIN genre_count g
CROSS JOIN recent_additions r;

Databricks data profile. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

## 🎯 How to Create Dashboards from These Queries

### Step 1: Run a Query
1. Execute any SQL cell above
2. Review the results table

### Step 2: Create Visualization
1. Click **"+ Visualization"** button above results
2. Choose visualization type:
   * **Bar Chart** - Comparisons (genres, countries)
   * **Line Chart** - Trends over time
   * **Pie Chart** - Distribution (ratings, percentages)
   * **Area Chart** - Cumulative growth
   * **Scatter Plot** - Multi-dimensional analysis

### Step 3: Configure Chart
1. **X-Axis**: Select dimension (e.g., genre, year)
2. **Y-Axis**: Select metric (e.g., content_count, avg_duration)
3. **Series/Color**: Group by another dimension
4. **Filters**: Add WHERE clauses to focus data
5. **Labels**: Add titles and axis labels

### Step 4: Add to Dashboard
1. Save the visualization
2. Click **"Add to Dashboard"**
3. Create new dashboard or add to existing
4. Arrange visualizations in layout

---

## 💡 Dashboard Best Practices

### Layout Recommendations:
1. **Top Section**: Executive KPIs (Query 10)
2. **Middle Section**: Trend charts (Queries 2, 3, 8)
3. **Bottom Section**: Detail analysis (Queries 1, 4, 5, 6)

### Performance Tips:
* **Use Gold tables** - Pre-aggregated for speed
* **Add filters** - Let users drill down by year, type, country
* **Cache results** - Enable result caching for faster loads
* **Optimize tables** - Already done with OPTIMIZE command

### Common Dashboard Layouts:

**1. Executive Dashboard:**
* KPIs (Query 10)
* Content growth trend (Query 2)
* Top genres (Query 1)
* Geographic distribution map (Query 4)

**2. Content Strategy Dashboard:**
* Movies vs TV Shows trend (Query 3)
* Genre diversity (Query 9)
* Rating distribution (Query 5)
* Decade analysis (Query 8)

**3. Operations Dashboard:**
* Duration analysis (Queries 6, 7)
* Country production (Query 4)
* Recent additions
* Quality metrics

---

## 🚀 Next Steps for Production

### Suggested Enhancements:

1. **Incremental Loading**
   * Change Bronze from `overwrite` to `append` mode
   * Add watermarking for date_added
   * Only process new records

2. **Data Quality Monitoring**
   * Add validation tables to track quality over time
   * Create alerts for anomalies
   * Log failed records for review

3. **Partitioning Strategy**
   * Partition Silver by `year_added`
   * Partition Gold by `decade` or `year`
   * Improves query performance for time-based filters

4. **Workflow Orchestration**
   * Create Databricks Job/Workflow
   * Schedule: Daily at 2 AM
   * Tasks: Bronze → Silver → Gold (sequential)
   * Add alerts for failures

5. **Access Control**
   * Grant SELECT on Gold to BI users
   * Grant MODIFY on Bronze/Silver to data engineers
   * Use Unity Catalog for fine-grained permissions

6. **Cost Optimization**
   * Enable Predictive Optimization (auto-optimize/vacuum)
   * Use Photon engine for faster queries
   * Set retention policies (VACUUM after 30 days)

7. **Monitoring & Observability**
   * Track row counts per layer
   * Monitor data freshness
   * Log transformation errors
   * Create SLA alerts

# ✅ Netflix Medallion Pipeline Complete!

## What We Built:

### 🥉 Bronze Layer
* Raw data ingestion from CSV
* Audit columns (ingestion_timestamp, source_file)
* Immutable source of truth
* Table: `workspace.netflix_bronze.raw_titles`

### 🥈 Silver Layer
* Cleaned and standardized data
* Null handling and deduplication
* Type conversions (dates, numbers)
* Derived columns (content_age, decade, duration_numeric)
* Data quality validations
* Table: `workspace.netflix_silver.clean_titles`

### 🥇 Gold Layer (6 Analytics Tables)
1. `workspace.netflix_gold.top_genres` - Genre analysis
2. `workspace.netflix_gold.content_trend_by_year` - Growth trends
3. `workspace.netflix_gold.duration_by_genre` - Duration insights
4. `workspace.netflix_gold.content_by_country` - Geographic distribution
5. `workspace.netflix_gold.content_by_rating` - Rating analysis
6. `workspace.netflix_gold.content_by_decade` - Historical trends

### 📊 Dashboard Queries
* 10 ready-to-use SQL queries
* Optimized for visualizations
* Business-focused insights

---

## 📚 Key Concepts Learned:

1. **Medallion Architecture** - Bronze/Silver/Gold layer separation
2. **Delta Lake** - ACID transactions, time travel, optimization
3. **Unity Catalog** - 3-level namespace, governed data
4. **PySpark** - Distributed data processing
5. **Data Quality** - Validation and cleansing patterns
6. **Performance** - Partitioning, OPTIMIZE, Z-ORDER
7. **Analytics** - Pre-aggregation for dashboard performance

---

## 🚀 Run the Pipeline:

```python
# Execute notebooks in order:
1. dbutils.notebook.run("01_Bronze_Netflix_Ingestion", timeout_seconds=600)
2. dbutils.notebook.run("02_Silver_Netflix_Transformation", timeout_seconds=600)
3. dbutils.notebook.run("03_Gold_Netflix_Analytics", timeout_seconds=600)
4. # Then use this notebook for dashboards
```

---

## 💼 Production Deployment:

Ready to schedule as a Databricks Job:
* Bronze → Silver → Gold tasks (sequential)
* Schedule: Daily/Hourly
* Alerts on failure
* Auto-scaling compute

**Congratulations on building a production-ready data pipeline!**